In [1]:
# --- Cell 1: Smart Offline Install (SAFE & MINIMAL) ---
import sys, os, glob, subprocess

print("⚙️ Smart offline install (minimal + compatible wheels only)...")

FORBIDDEN = ("torch", "torchvision", "torchaudio", "numpy", "pandas", "opencv", "matplotlib", "scipy", "pillow")
NEEDED_HINTS = ("segmentation_models_pytorch", "ultralytics", "timm")

py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"   # cp311
print("🐍 Python tag:", py_tag)

all_whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
print(f"📦 Found wheels: {len(all_whls)}")

def ok_whl(name: str) -> bool:
    n = name.lower()
    if any(x in n for x in FORBIDDEN):
        return False
    return ("py3-none-any" in n) or (py_tag in n)

picked = []
for p in all_whls:
    n = os.path.basename(p)
    if any(h in n.lower() for h in NEEDED_HINTS) and ok_whl(n):
        picked.append(p)

picked = sorted(set(picked), key=len)
print("✅ Will try install:", len(picked), "wheels")

ok = 0
for whl in picked:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", whl, "--no-deps", "--ignore-installed", "--quiet"])
        ok += 1
    except:
        print("⚠️ Failed:", os.path.basename(whl))

print(f"✅ Installed {ok}/{len(picked)} wheels safely.")

import torch
print("⚡ Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


⚙️ Smart offline install (minimal + compatible wheels only)...
🐍 Python tag: cp311
📦 Found wheels: 95
✅ Will try install: 3 wheels
✅ Installed 3/3 wheels safely.
⚡ Torch: 2.6.0+cu124 | CUDA: True
GPU: Tesla T4


In [2]:
# --- Cell 2: Offline install/import fix (NO internet) + Path Resolver ---
import sys, os, glob, subprocess, site
from importlib import reload

print("🔧 Cell 2: Offline install/import fix (no internet).")

# تعطيل أي شيء ممكن يحاول يتصل بالنت
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def ensure_pkg(import_name: str, wheel_hint: str):
    """Try import; if missing install from /kaggle/input wheels offline."""
    try:
        __import__(import_name)
        print(f"✅ Already available: {import_name}")
        return True
    except Exception:
        wheels = glob.glob(f"/kaggle/input/**/*{wheel_hint}*.whl", recursive=True)
        if not wheels:
            print(f"❌ Missing wheel for: {import_name} (hint={wheel_hint})")
            return False
        target = sorted(wheels, key=len)[0]
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", target, "--no-deps", "--ignore-installed", "--quiet"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
            reload(site)
            __import__(import_name)
            print(f"✅ Installed offline: {import_name}")
            return True
        except Exception as e:
            print(f"⚠️ Offline install failed for {import_name}: {e}")
            return False

ensure_pkg("timm", "timm")
ensure_pkg("segmentation_models_pytorch", "segmentation_models_pytorch")
ensure_pkg("ultralytics", "ultralytics")

# محرك parquet
try:
    import pyarrow  # noqa
    print("✅ pyarrow available (parquet OK)")
except Exception:
    print("⚠️ pyarrow not found. If parquet read fails later, Kaggle environment may be missing parquet engine.")

import torch
import segmentation_models_pytorch as smp

print("------ Environment Check ------")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("smp:", getattr(smp, "__version__", "unknown"))

# ---------- PATHS (عدل هنا فقط عند الحاجة) ----------
NEW_UNET_PATH = "/kaggle/input/best-dd4/best_model_effb3_phase9_ddp.pth"
OLD_UNET_PATH = "/kaggle/input/ecg-final-models1/best_model_phase10.pth"
YOLO_PATH     = "/kaggle/input/ecg-final-models1/best.pt"

print("📌 Resolved paths:")
print(" - NEW_UNET_PATH:", NEW_UNET_PATH, "| exists:", os.path.exists(NEW_UNET_PATH))
print(" - OLD_UNET_PATH:", OLD_UNET_PATH, "| exists:", os.path.exists(OLD_UNET_PATH))
print(" - YOLO_PATH    :", YOLO_PATH,     "| exists:", os.path.exists(YOLO_PATH))

print("✅ Cell 2 ready.")


🔧 Cell 2: Offline install/import fix (no internet).


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ Already available: timm
✅ Installed offline: segmentation_models_pytorch
✅ Already available: ultralytics
✅ pyarrow available (parquet OK)
------ Environment Check ------
torch: 2.6.0+cu124
cuda available: True
gpu: Tesla T4
smp: 0.5.0
📌 Resolved paths:
 - NEW_UNET_PATH: /kaggle/input/best-dd4/best_model_effb3_phase9_ddp.pth | exists: True
 - OLD_UNET_PATH: /kaggle/input/ecg-final-models1/best_model_phase10.pth | exists: True
 - YOLO_PATH    : /kaggle/input/ecg-final-models1/best.pt | exists: True
✅ Cell 2 ready.


In [3]:
# --- Cell 2.5: Attach NEW trained model (EffB3) to inference ---
import os, shutil

# عدّل لو عندك مسار مختلف
NEW_BEST_SRC = "/kaggle/input/best-dd4/best_model_effb3_phase9_ddp.pth"

# إذا كنت مخزنها في Dataset input بدل working ضع المسار هنا:
# NEW_BEST_SRC = "/kaggle/input/<your-dataset>/best_model_effb3_phase9_ddp.pth"

NEW_BEST_DST = "/kaggle/input/best-dd4/best_model_effb3_phase9_ddp.pth"

if os.path.exists(NEW_BEST_SRC):
    if NEW_BEST_SRC != NEW_BEST_DST:
        shutil.copy(NEW_BEST_SRC, NEW_BEST_DST)
    print("✅ NEW model ready:", NEW_BEST_DST)
else:
    print("❌ NEW model not found at:", NEW_BEST_SRC)

print("Exists:", os.path.exists(NEW_BEST_DST))


✅ NEW model ready: /kaggle/input/best-dd4/best_model_effb3_phase9_ddp.pth
Exists: True


In [4]:
# # --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# # إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
# DB_DIR = "physionet_db"
# if not os.path.exists(DB_DIR):
#     os.makedirs(DB_DIR)
#     print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
#     try:
#         # تحميل عينة كافية لضمان التنوع
#         records = wfdb.get_record_list('ptb-xl')
#         selected_records = records[:200] 
#         wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
#         print(f"✅ تم تحميل {len(selected_records)} سجل.")
#     except Exception as e:
#         print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

# class UltimateRenderer:
#     def __init__(self, db_dir):
#         self.db_dir = db_dir
#         self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
#     def get_real_signal(self):
#         """سحب إشارة عشوائية من PTB-XL"""
#         if not self.records:
#             t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
#         try:
#             rec_name = random.choice(self.records)
#             record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
#             lead_idx = random.randint(0, record.shape[1] - 1)
#             signal = record[:, lead_idx]
#             signal = np.nan_to_num(signal)
            
#             # قص عشوائي (Zoom in/out simulation)
#             crop_len = random.randint(1000, 3000)
#             if len(signal) > crop_len:
#                 start = random.randint(0, len(signal) - crop_len)
#                 return signal[start : start+crop_len]
#             return signal
#         except:
#             return np.random.randn(2000)

#     def render_to_memory(self, signal):
#         """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
#         # 3. الشبكة المتغيرة (Distractor)
#         grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
#         grid_alpha = random.uniform(0.3, 0.8)
#         bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
#         fig_h, fig_w = 3, 8
#         dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
#         # --- أ. توليد القناع (Mask Generation) ---
#         fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
#         ax.plot(signal, color='white', linewidth=3.0) 
#         ax.set_ylim(np.min(signal), np.max(signal))
#         ax.axis('off')
#         fig.patch.set_facecolor('black')
        
#         fig.canvas.draw()
#         mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
#         mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
#         mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
#         _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
#         plt.close(fig)

#         # --- ب. الرسم الأولي (Rendering) ---
#         fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
#         ax.set_facecolor(bg_color)
        
#         # رسم الشبكة
#         ax.minorticks_on()
#         ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
#         ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
#         # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
#         line_color = random.choice(['black', 'blue', '#000033']) 
#         ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
#         ax.axis('off')
#         ax.set_ylim(np.min(signal), np.max(signal))
#         fig.patch.set_facecolor(bg_color)
        
#         fig.canvas.draw()
#         img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
#         img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
#         # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
#         img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
#         plt.close(fig)
        
#         return img, mask

# print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")
# --- Cell 3: (Disabled in Submission) Renderer/PTB-XL not needed ---
print("✅ Cell 3 disabled: Renderer/PTB-XL not needed for submission.")


✅ Cell 3 disabled: Renderer/PTB-XL not needed for submission.


In [5]:
# --- NEW DIAGNOSTIC CELL: Check real coverage of pid->path mapping ---

import os, glob, re
import pandas as pd
from collections import OrderedDict

IMAGE_DIR = "/kaggle/input/physionet-ecg-image-digitization"
SAMPLE_PARQUET_PATH = f"{IMAGE_DIR}/sample_submission.parquet"

def clean_pid(pid):
    s = str(pid).strip()
    return s[:-2] if s.endswith(".0") else s

def parse_rid(rid: str):
    parts = rid.rsplit("_", 2)
    if len(parts) != 3:
        return None
    pid_raw, idx_raw, lead = parts
    return pid_raw

# 1) load template ids
tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
template_ids = tmpl["id"].astype(str).to_list()
del tmpl

# 2) collect up to 200 UNIQUE pids by scanning across whole template (not first 300)
seen = OrderedDict()
for rid in template_ids:
    pid = parse_rid(rid)
    if pid is None:
        continue
    if pid not in seen:
        seen[pid] = True
    if len(seen) >= 200:
        break
pids = list(seen.keys())
print("✅ Unique PIDs sampled:", len(pids))

# 3) build smart index
def first_int(s):
    m = re.search(r"\d+", s)
    return m.group(0) if m else None

image_paths = glob.glob(f"{IMAGE_DIR}/**/*.png", recursive=True) + glob.glob(f"{IMAGE_DIR}/**/*.jpg", recursive=True)
pid_to_path = {}

for p in image_paths:
    base = os.path.splitext(os.path.basename(p))[0].strip()
    pid_to_path[base] = p
    pid_to_path[clean_pid(base)] = p
    num = first_int(base)
    if num is not None:
        pid_to_path[num.lstrip("0") or "0"] = p

def get_path(pid_raw):
    pid = clean_pid(pid_raw)
    p = pid_to_path.get(pid)
    if p and os.path.exists(p):
        return p
    try:
        k = str(int(float(pid)))
        p = pid_to_path.get(k)
        if p and os.path.exists(p):
            return p
    except:
        pass
    return None

# 4) coverage test
hits = 0
miss = []
for pid in pids:
    if get_path(pid):
        hits += 1
    else:
        if len(miss) < 25:
            miss.append(pid)

ratio = hits / max(1, len(pids))
print(f"🧪 Coverage: {hits}/{len(pids)} = {ratio*100:.1f}%")
print("❌ Missing PID examples (up to 25):", miss)

# 5) show some real filenames to infer pattern
print("\n📌 Example image basenames (first 25):")
for p in image_paths[:25]:
    print(" -", os.path.splitext(os.path.basename(p))[0])


✅ Unique PIDs sampled: 2
🧪 Coverage: 2/2 = 100.0%
❌ Missing PID examples (up to 25): []

📌 Example image basenames (first 25):
 - 2352854581
 - 1053922973
 - 735384893-0005
 - 735384893-0006
 - 735384893-0011
 - 735384893-0004
 - 735384893-0012
 - 735384893-0003
 - 735384893-0001
 - 735384893-0009
 - 735384893-0010
 - 2591785253-0005
 - 2591785253-0003
 - 2591785253-0001
 - 2591785253-0006
 - 2591785253-0012
 - 2591785253-0011
 - 2591785253-0004
 - 2591785253-0009
 - 2591785253-0010
 - 2617381459-0006
 - 2617381459-0009
 - 2617381459-0005
 - 2617381459-0012
 - 2617381459-0003


In [6]:
# --- Cell 22 (REPLACEMENT): V42 Robust Inference (Deskew + Trim + Model-Select + Quality Gate) ---

import gc, os, csv, glob, math
import numpy as np
import pandas as pd
import cv2
import torch
from collections import OrderedDict
from tqdm import tqdm
from scipy.signal import savgol_filter, resample, butter, filtfilt
import segmentation_models_pytorch as smp
from ultralytics import YOLO

# ---------------------------
# 0) Constants
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_CSV_PATH = "/kaggle/input/physionet-ecg-image-digitization/test.csv"
SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"
IMAGE_DIR = "/kaggle/input/physionet-ecg-image-digitization"

# paths come from Cell 2
assert "YOLO_PATH" in globals() and "NEW_UNET_PATH" in globals() and "OLD_UNET_PATH" in globals(), "❌ Run Cell 2 first."

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
LEAD_TO_IDX = {name: i for i, name in enumerate(LEAD_NAMES)}

TARGET_H = 256
YOLO_INF_MAX = 1280
YOLO_CONF = 0.08

MAX_W = 2048
MAX_CACHE = 8

DP_MAX_W = 1500
DP_SMOOTH = 0.45

print(f"⚡ Device: {DEVICE}")
print(f"📌 Paths exist: | NEW: {os.path.exists(NEW_UNET_PATH)} | OLD: {os.path.exists(OLD_UNET_PATH)} | YOLO: {os.path.exists(YOLO_PATH)}")

# ---------------------------
# 1) Read template
# ---------------------------
tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
template_ids = tmpl["id"].astype(str).to_numpy()
del tmpl
gc.collect()
print(f"✅ Template rows: {len(template_ids)}")

# ---------------------------
# 2) Build patient_lengths (prefer test.csv number_of_rows if exists)
# ---------------------------
patient_lengths = {}
fs_map = {}

if os.path.exists(TEST_CSV_PATH):
    tdf = pd.read_csv(TEST_CSV_PATH, dtype=str)
    cols_low = {c.lower(): c for c in tdf.columns}
    id_c = next((cols_low[c] for c in cols_low if c == "id" or "id" in c), None)
    fs_c = next((cols_low[c] for c in cols_low if "fs" in c), None)
    nr_c = next((cols_low[c] for c in cols_low if "number_of_rows" in c or "rows" in c), None)

    if id_c and fs_c:
        fs_map = dict(zip(tdf[id_c].astype(str), tdf[fs_c].astype(str)))
    if id_c and nr_c:
        # number_of_rows غالباً ثابت لكل PID
        for pid, grp in tdf.groupby(id_c):
            try:
                patient_lengths[str(pid)] = int(float(grp[nr_c].iloc[0]))
            except:
                pass

    print("✅ test.csv columns:", list(tdf.columns))
    print(f"✅ fs_map: {len(fs_map)} | patient_lengths(from test.csv): {len(patient_lengths)}")

# fallback: parse from template ids
if len(patient_lengths) == 0:
    print("🧠 Building patient lengths from template ids (fallback)...")
    for rid in template_ids:
        try:
            parts = rid.rsplit("_", 2)  # pid_idx_lead
            if len(parts) != 3:
                continue
            pid = str(parts[0]).strip()
            idx = int(parts[1])
            patient_lengths[pid] = max(patient_lengths.get(pid, 0), idx + 1)
        except:
            pass
    print(f"✅ patient_lengths(from template): {len(patient_lengths)}")

# ---------------------------
# 3) Index images
# ---------------------------
print("🗂️ Indexing images...")
image_paths = glob.glob(f"{IMAGE_DIR}/**/*.png", recursive=True) + glob.glob(f"{IMAGE_DIR}/**/*.jpg", recursive=True)
# key = basename without extension
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
print(f"✅ Indexed images: {len(path_map)}")

# ---------------------------
# 4) Load YOLO + models
# ---------------------------
print("⚙️ Loading YOLO (offline)...")
yolo_model = YOLO(YOLO_PATH)
names = getattr(yolo_model, "names", None)

CLASS_TO_LEAD_IDX = {}
if isinstance(names, dict):
    items = list(names.items())
elif isinstance(names, list):
    items = list(enumerate(names))
else:
    items = []

def _normalize_lead_name(x: str) -> str:
    s = str(x).strip()
    s = s.replace("Lead", "").replace("lead", "").replace("_", "").replace("-", "").replace(" ", "")
    s = s.replace("AVR", "aVR").replace("AVL", "aVL").replace("AVF", "aVF")
    s = s.replace("AVr","aVR").replace("AVl","aVL").replace("AVf","aVF")
    return s

for cid, cname in items:
    n = _normalize_lead_name(cname)
    if n in LEAD_TO_IDX:
        CLASS_TO_LEAD_IDX[int(cid)] = LEAD_TO_IDX[n]

# إذا mapping ناقص، خليها identity 0..11
for i in range(12):
    CLASS_TO_LEAD_IDX.setdefault(i, i)

print("✅ YOLO loaded:", YOLO_PATH)
print("✅ CLASS_TO_LEAD_IDX:", CLASS_TO_LEAD_IDX)

print("⚙️ Loading UNet models (offline)...")

# ملاحظة: نحن نفترض أن الاثنين UNet-resnet50 (مثل اللي عندك نجح تحميله)
def build_unet_resnet50():
    return smp.Unet(
        encoder_name="resnet50",
        encoder_weights=None,
        in_channels=3,
        classes=1,
        decoder_attention_type="scse",
    )

new_model = build_unet_resnet50().to(DEVICE).eval()
old_model = build_unet_resnet50().to(DEVICE).eval()

# تحميل NEW
new_model.load_state_dict(torch.load(NEW_UNET_PATH, map_location=DEVICE))
print("✅ NEW loaded")

# تحميل OLD
old_model.load_state_dict(torch.load(OLD_UNET_PATH, map_location=DEVICE))
print("✅ OLD loaded (resnet50)")

# ---------------------------
# 5) Helpers
# ---------------------------
def clean_pid(pid):
    s = str(pid).strip()
    return s[:-2] if s.endswith(".0") else s

def get_image_path(pid):
    pid_s = clean_pid(pid)
    if pid_s in path_map:
        return path_map[pid_s]
    # محاولة رقمية
    try:
        return path_map.get(str(int(float(pid_s))))
    except:
        return None

def safe_read_rgb(path):
    img = cv2.imread(path)
    if img is None:
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def rotate_keep_white(img_rgb, angle_deg):
    h, w = img_rgb.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle_deg, 1.0)
    return cv2.warpAffine(img_rgb, M, (w, h), flags=cv2.INTER_LINEAR, borderValue=(255,255,255))

def estimate_skew_angle(img_rgb):
    """
    Deskew using Hough lines (robust, light). Returns angle in degrees to rotate.
    """
    try:
        gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        gray = cv2.GaussianBlur(gray, (5,5), 0)
        edges = cv2.Canny(gray, 50, 150, apertureSize=3)
        # Hough lines
        lines = cv2.HoughLines(edges, 1, np.pi/180, 200)
        if lines is None:
            return 0.0
        angles = []
        for l in lines[:200]:
            rho, theta = l[0]
            # line angle = theta - 90deg
            ang = (theta - np.pi/2.0) * 180.0/np.pi
            if -30 <= ang <= 30:
                angles.append(ang)
        if len(angles) < 5:
            return 0.0
        return float(np.median(angles))
    except:
        return 0.0

def preprocess_remove_grid_rgb(img_rgb):
    # remove red/pink grid lightly
    try:
        hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
        mask = (cv2.inRange(hsv, (0, 50, 50), (10, 255, 255)) |
                cv2.inRange(hsv, (170, 50, 50), (180, 255, 255)))
        out = img_rgb.copy()
        out[mask > 0] = (255, 255, 255)
        return out
    except:
        return img_rgb

def apply_high_pass(data, cutoff=0.5, fs=500.0, order=3):
    try:
        if len(data) < order * 3:
            return data
        nyq = 0.5 * fs
        if nyq <= 0:
            return data
        wn = cutoff / nyq
        if not (0 < wn < 1):
            return data
        b, a = butter(order, wn, btype="high")
        return filtfilt(b, a, data)
    except:
        return data

# ---------------------------
# 6) Viterbi / Path + Trim
# ---------------------------
def viterbi_dp_y(prob_map, smooth=0.45):
    H, W = prob_map.shape
    cost = -np.log(prob_map + 1e-6).astype(np.float32)
    dp = np.zeros_like(cost, dtype=np.float32)
    parent = np.zeros((H, W), dtype=np.int16)

    dp[:, 0] = cost[:, 0]
    parent[:, 0] = np.arange(H, dtype=np.int16)

    for t in range(1, W):
        prev = dp[:, t-1]
        c_m1 = np.roll(prev, 1);  c_m1[0] = 1e9
        c_0  = prev
        c_p1 = np.roll(prev, -1); c_p1[-1] = 1e9
        stacked = np.stack([c_m1 + smooth, c_0, c_p1 + smooth], axis=0)
        which = np.argmin(stacked, axis=0).astype(np.int16)
        best_prev = stacked[which, np.arange(H)]
        dp[:, t] = cost[:, t] + best_prev
        parent[:, t] = np.clip(np.arange(H, dtype=np.int16) + (which - 1), 0, H-1)

    path = np.zeros(W, dtype=np.int32)
    path[-1] = int(np.argmin(dp[:, -1]))
    for t in range(W-2, -1, -1):
        path[t] = int(parent[path[t+1], t+1])
    return path  # y index 0..H-1

def viterbi_adaptive_y(prob_map):
    H, W = prob_map.shape
    if W <= 20 or H <= 5:
        return np.argmax(prob_map, axis=0).astype(np.int32)
    if W <= DP_MAX_W:
        return viterbi_dp_y(prob_map, smooth=DP_SMOOTH)
    # fast path
    y = np.argmax(prob_map, axis=0).astype(np.float32)
    win = 21 if W >= 21 else (W if W % 2 == 1 else W-1)
    if win >= 5:
        y = savgol_filter(y, win, 2)
    return np.clip(y, 0, H-1).astype(np.int32)

def trim_prob_map(prob, pad=12):
    """
    Remove blank margins using column max.
    """
    col = prob.max(axis=0)
    thr = max(0.06, 0.25 * float(col.max()))
    idx = np.where(col > thr)[0]
    if len(idx) < 40:
        return prob  # لا تقص إذا ضعيف جداً
    s = max(int(idx[0]) - pad, 0)
    e = min(int(idx[-1]) + pad, prob.shape[1]-1)
    return prob[:, s:e+1]

def quality_score(prob):
    """
    Higher is better.
    """
    prob = trim_prob_map(prob)
    H, W = prob.shape
    y = viterbi_adaptive_y(prob)
    # prob along path
    p = prob[y, np.arange(W)]
    mean_p = float(np.mean(p))
    # roughness
    dy = np.diff(y).astype(np.float32)
    rough = float(np.mean(np.abs(dy)))
    # gaps: low confidence on path
    gap = float(np.mean(p < 0.10))
    return mean_p - 0.01*rough - 0.20*gap, mean_p

# ---------------------------
# 7) YOLO crops (after deskew)
# ---------------------------
def get_crops_yolo_mapped(img_rgb):
    crops = [None] * 12
    h0, w0 = img_rgb.shape[:2]
    if h0 < 20 or w0 < 20:
        return crops

    scale = YOLO_INF_MAX / float(max(h0, w0))
    if scale < 1.0:
        w1, h1 = max(32, int(w0 * scale)), max(32, int(h0 * scale))
        img_inf = cv2.resize(img_rgb, (w1, h1))
    else:
        img_inf = img_rgb
        scale = 1.0

    best = {}
    res = yolo_model.predict(img_inf, conf=YOLO_CONF, verbose=False, device=DEVICE)
    if res and res[0].boxes is not None and len(res[0].boxes) > 0:
        boxes = res[0].boxes.data.detach().cpu().numpy()
        for b in boxes:
            x1, y1, x2, y2, conf, cls_id = b[:6]
            cls_id = int(cls_id)
            conf = float(conf)
            lead_idx = CLASS_TO_LEAD_IDX.get(cls_id, cls_id)
            if not (0 <= lead_idx < 12):
                continue

            x1o, y1o, x2o, y2o = x1/scale, y1/scale, x2/scale, y2/scale
            x1o, y1o = max(0, int(x1o)), max(0, int(y1o))
            x2o, y2o = min(w0, int(x2o)), min(h0, int(y2o))
            if x2o <= x1o + 10 or y2o <= y1o + 10:
                continue

            prev = best.get(lead_idx)
            if (prev is None) or (conf > prev[0]):
                best[lead_idx] = (conf, (x1o, y1o, x2o, y2o))

    for li, (_, (x1o, y1o, x2o, y2o)) in best.items():
        crops[li] = img_rgb[y1o:y2o, x1o:x2o]
    return crops

# ---------------------------
# 8) UNet predict (for a given model)
# ---------------------------
def prepare_unet_batch(crops_rgb):
    tens_list, scales, widths = [], [], []
    for c in crops_rgb:
        h, w = c.shape[:2]
        if h < 5 or w < 5:
            continue
        sc = TARGET_H / float(h)
        nw = int(max(1, w * sc))
        if nw > MAX_W:
            nw = MAX_W
            sc = nw / float(w)
        rz = cv2.resize(c, (nw, TARGET_H))
        t = torch.from_numpy(rz).permute(2,0,1).float() / 255.0
        tens_list.append(t)
        scales.append(sc)
        widths.append(nw)

    if not tens_list:
        return None, None, None

    max_w = int(np.ceil(max(widths) / 32.0) * 32)
    max_w = min(max_w, MAX_W)

    batch = torch.zeros((len(tens_list), 3, TARGET_H, max_w), dtype=torch.float32, device=DEVICE)
    for i, t in enumerate(tens_list):
        w_i = min(t.shape[2], max_w)
        batch[i, :, :, :w_i] = t[:, :, :w_i].to(DEVICE)

    return batch, scales, widths

def predict_probmaps(model, crops_rgb):
    batch, scales, widths = prepare_unet_batch(crops_rgb)
    if batch is None:
        return [], []

    with torch.no_grad():
        if DEVICE == "cuda":
            with torch.amp.autocast("cuda", enabled=True):
                p1 = torch.sigmoid(model(batch))
                p2 = torch.sigmoid(model(torch.flip(batch, [3])))
                preds = (p1 + torch.flip(p2, [3])) / 2.0
        else:
            p1 = torch.sigmoid(model(batch))
            p2 = torch.sigmoid(model(torch.flip(batch, [3])))
            preds = (p1 + torch.flip(p2, [3])) / 2.0

    preds = preds.detach().float().cpu().numpy()  # [N,1,H,W]
    out = []
    for i in range(preds.shape[0]):
        w_i = widths[i]
        out.append(preds[i,0,:,:w_i].astype(np.float32))
    return out, scales

# ---------------------------
# 9) Convert probmap -> mV signal (safe)
# ---------------------------
def prob_to_signal_mv(prob, fs, target_len):
    prob = trim_prob_map(prob)
    H, W = prob.shape
    y = viterbi_adaptive_y(prob)
    # convert y->pixel amplitude (invert)
    raw_px = (H - 1 - y).astype(np.float32)

    # simple scaling fallback (do NOT overfit grid detection هنا)
    # لأن بعض الصور بدون grid واضح، والbaseline صفر أفضل من scaling خاطئ جداً
    ppmv = 250.0  # safe fallback

    sig = (raw_px - np.median(raw_px)) / ppmv
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)

    if len(sig) >= 11:
        sig = savgol_filter(sig, 11, 3)
    if len(sig) >= 30:
        sig = apply_high_pass(sig, cutoff=0.5, fs=fs, order=3)

    sig = np.clip(sig, -7.0, 7.0)

    # resample to target_len
    if len(sig) > 0 and target_len > 0:
        sig = resample(sig, target_len).astype(np.float32)
    else:
        sig = np.zeros((target_len,), dtype=np.float32)

    return sig

# ---------------------------
# 10) Patient compute with Deskew + Model Select + Quality Gate
# ---------------------------
patient_cache = OrderedDict()

def compute_patient_leads(pid, target_len):
    out = np.zeros((12, target_len), dtype=np.float32)

    path = get_image_path(pid)
    if not path:
        return out
    img = safe_read_rgb(path)
    if img is None:
        return out

    # Deskew before YOLO (مهم جداً في الصور المائلة)
    ang = estimate_skew_angle(img)
    if abs(ang) >= 0.4:
        img = rotate_keep_white(img, -ang)

    fs = float(fs_map.get(clean_pid(pid), 500.0))

    crops = get_crops_yolo_mapped(img)
    detected = [(i, c) for i, c in enumerate(crops) if c is not None and c.size > 200]
    if not detected:
        return out

    lead_indices = [i for i, _ in detected]
    crops_rgb = [preprocess_remove_grid_rgb(c) for _, c in detected]

    # predict from both models
    prob_new, _ = predict_probmaps(new_model, crops_rgb)
    prob_old, _ = predict_probmaps(old_model, crops_rgb)

    # choose best per lead (or fallback to zeros)
    for j in range(len(crops_rgb)):
        li = lead_indices[j]

        if j >= len(prob_new) or j >= len(prob_old):
            continue

        sN, mN = quality_score(prob_new[j])
        sO, mO = quality_score(prob_old[j])

        # select better
        prob = prob_new[j] if sN >= sO else prob_old[j]
        best_mean_p = mN if sN >= sO else mO

        # Quality Gate:
        # إذا الثقة ضعيفة جداً، رجّع zeros (baseline أفضل من garbage) :contentReference[oaicite:3]{index=3}
        if best_mean_p < 0.08:
            out[li] = np.zeros((target_len,), dtype=np.float32)
            continue

        sig = prob_to_signal_mv(prob, fs, target_len)

        # amplitude gate (إذا ضعيفة جداً أو مجنونة جداً)
        p2p = float(np.percentile(sig, 99) - np.percentile(sig, 1))
        if p2p < 0.02 or p2p > 8.5:
            out[li] = np.zeros((target_len,), dtype=np.float32)
        else:
            out[li] = sig

    return out

# ---------------------------
# 11) Write submission
# ---------------------------
print("🚀 Writing submission.csv (V42 Robust)...")

with open(SUBMISSION_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "value"])

    for rid in tqdm(template_ids, desc="Processing Rows"):
        try:
            parts = rid.rsplit("_", 2)
            if len(parts) != 3:
                writer.writerow([rid, "0.0000"])
                continue

            pid_raw, idx_raw, lead_name = parts
            pid = clean_pid(pid_raw)
            idx = int(idx_raw)

            t_len = patient_lengths.get(pid, None)
            if t_len is None:
                # fallback: infer from ids
                t_len = patient_lengths.get(pid_raw, 5000)
            if t_len <= 0 or t_len > 20000:
                t_len = 5000

            if pid in patient_cache:
                mtx = patient_cache[pid]
                patient_cache.move_to_end(pid)
            else:
                mtx = compute_patient_leads(pid, t_len)
                patient_cache[pid] = mtx
                if len(patient_cache) > MAX_CACHE:
                    patient_cache.popitem(last=False)

            lead_idx = LEAD_TO_IDX.get(lead_name, 0)

            val = 0.0
            if 0 <= idx < mtx.shape[1]:
                v = float(mtx[lead_idx][idx])
                if np.isfinite(v):
                    val = v

            writer.writerow([rid, f"{val:.4f}"])
        except:
            writer.writerow([rid, "0.0000"])

del patient_cache
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("✅ Done. submission.csv ready.")


⚡ Device: cuda
📌 Paths exist: | NEW: True | OLD: True | YOLO: True
✅ Template rows: 75000
✅ test.csv columns: ['id', 'lead', 'fs', 'number_of_rows']
✅ fs_map: 2 | patient_lengths(from test.csv): 2
🗂️ Indexing images...
✅ Indexed images: 8795
⚙️ Loading YOLO (offline)...
✅ YOLO loaded: /kaggle/input/ecg-final-models1/best.pt
✅ CLASS_TO_LEAD_IDX: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11}
⚙️ Loading UNet models (offline)...


RuntimeError: Error(s) in loading state_dict for Unet:
	Missing key(s) in state_dict: "encoder.conv1.weight", "encoder.bn1.weight", "encoder.bn1.bias", "encoder.bn1.running_mean", "encoder.bn1.running_var", "encoder.layer1.0.conv1.weight", "encoder.layer1.0.bn1.weight", "encoder.layer1.0.bn1.bias", "encoder.layer1.0.bn1.running_mean", "encoder.layer1.0.bn1.running_var", "encoder.layer1.0.conv2.weight", "encoder.layer1.0.bn2.weight", "encoder.layer1.0.bn2.bias", "encoder.layer1.0.bn2.running_mean", "encoder.layer1.0.bn2.running_var", "encoder.layer1.0.conv3.weight", "encoder.layer1.0.bn3.weight", "encoder.layer1.0.bn3.bias", "encoder.layer1.0.bn3.running_mean", "encoder.layer1.0.bn3.running_var", "encoder.layer1.0.downsample.0.weight", "encoder.layer1.0.downsample.1.weight", "encoder.layer1.0.downsample.1.bias", "encoder.layer1.0.downsample.1.running_mean", "encoder.layer1.0.downsample.1.running_var", "encoder.layer1.1.conv1.weight", "encoder.layer1.1.bn1.weight", "encoder.layer1.1.bn1.

In [ ]:
# =========================
# --- الخلية 23: Strict Submission Validator (Must Pass) ---
# =========================
import os
import numpy as np
import pandas as pd

SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

print("🔍 Validating submission.csv strictly...")

if not os.path.exists(SUBMISSION_FILE):
    raise FileNotFoundError("❌ submission.csv not found. Run Cell 22 first.")

# اقرأ القالب (ids فقط)
tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
tmpl_ids = tmpl["id"].astype(str).to_numpy()
n_tmpl = len(tmpl_ids)

# اقرأ ملفك
sub = pd.read_csv(SUBMISSION_FILE)

# 1) الأعمدة
assert list(sub.columns) == ["id", "value"], f"❌ Columns mismatch: {sub.columns}"

# 2) عدد الصفوف
assert len(sub) == n_tmpl, f"❌ Row count mismatch: sub={len(sub)} vs template={n_tmpl}"

# 3) عدم وجود NaN
assert sub["id"].isna().sum() == 0, "❌ Found NaN in id"
assert sub["value"].isna().sum() == 0, "❌ Found NaN in value"

# 4) تحويل value إلى float والتأكد finite
vals = pd.to_numeric(sub["value"], errors="coerce").to_numpy()
assert np.isfinite(vals).all(), "❌ Found non-finite values (NaN/inf) in value"

# 5) مطابقة أول وآخر ID (كفاية لاكتشاف تغيير ترتيب/تنظيف خاطئ)
sub_ids = sub["id"].astype(str).to_numpy()
assert sub_ids[0] == tmpl_ids[0], f"❌ First ID mismatch: {sub_ids[0]} != {tmpl_ids[0]}"
assert sub_ids[-1] == tmpl_ids[-1], f"❌ Last ID mismatch: {sub_ids[-1]} != {tmpl_ids[-1]}"

# 6) فحص عينة عشوائية للمطابقة (بدون مقارنة كل شيء لتوفير وقت)
idxs = np.linspace(0, n_tmpl-1, num=min(20, n_tmpl), dtype=int)
for i in idxs:
    if sub_ids[i] != tmpl_ids[i]:
        raise AssertionError(f"❌ ID mismatch at row {i}: {sub_ids[i]} != {tmpl_ids[i]}")

size_mb = os.path.getsize(SUBMISSION_FILE) / (1024*1024)
print("✅ All checks passed.")
print(f"📦 submission.csv size: {size_mb:.2f} MB")
print("🎉 Ready to Submit.")
